# KolektorSDD1 ResNet18-UNet Fine-Tuning

This notebook trains a ResNet18-UNet segmentation model on KolektorSDD1 for crack detection. It loads the dataset, builds a stratified train/validation split, applies data augmentation, trains the selected fine-tuning strategy, tracks segmentation metrics, and visualizes predictions and threshold behavior.

The model uses segmentation-models-pytorch with a ResNet18 encoder and UNet decoder. The main workflow includes loss/metric tracking, checkpoint selection, learning-curve plotting, qualitative prediction inspection, and threshold sweeping.


## Environment Setup

Install the PyTorch, segmentation-models-pytorch, image-processing, and utility dependencies needed by the notebook.


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install segmentation-models-pytorch pillow tqdm opencv-python scikit-learn

## Dataset Extraction

Create the Colab dataset folder and unzip the KolektorSDD1 archive into the expected location.


In [ ]:
!mkdir -p /content/KolektorSDD-boxes && unzip -q /content/KolektorSDD-boxes.zip -d /content/KolektorSDD-boxes

## Imports

Load the Python, PyTorch, plotting, image-processing, model, splitting, and progress-bar libraries used throughout the notebook.


In [ ]:
# ─────────────────────────────────────────────
# 1. Imports
# ─────────────────────────────────────────────
import os
import random
import time

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torch.optim.lr_scheduler import CosineAnnealingLR

import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from tqdm import tqdm

## Configuration

Define dataset paths, image size, training hyperparameters, thresholding, loss settings, random seeds, and output options.


In [ ]:
# ─────────────────────────────────────────────
# 2. Config
# ─────────────────────────────────────────────
DATA_ROOT    = "/content/KolektorSDD-boxes"
IMG_SIZE     = (512, 1408)
BATCH_SIZE   = 1
NUM_EPOCHS   = 50
LR           = 1e-4
WEIGHT_DECAY = 1e-4
THR          = 0.6

POS_WEIGHT   = 10.0
LOSS_MODE    = "focal"   # "bce_dice" or "focal_dice" or "focal"
FOCAL_ALPHA  = 0.75
FOCAL_GAMMA  = 2.0

SPLIT_SEED   = 42
RUN_SEEDS    = [11]

MODEL_SELECTION_LAMBDA = 0.10   # score = pos_dice - lambda * neg_fp_rate

SAVE_HISTORY = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SPLIT_SEED)

## Dataset Loading

Collect image/mask pairs, load and resize samples, apply optional augmentation, normalize images, and expose them through a PyTorch dataset.


In [ ]:
# ─────────────────────────────────────────────
# 3. Dataset
# ─────────────────────────────────────────────
def collect_samples(data_root):
    """Walk DATA_ROOT and collect (image_path, mask_path) pairs."""
    samples = []
    for kos_folder in sorted(os.listdir(data_root)):
        kos_path = os.path.join(data_root, kos_folder)
        if not os.path.isdir(kos_path):
            continue
        for fname in os.listdir(kos_path):
            if fname.endswith(".jpg"):
                img_path  = os.path.join(kos_path, fname)
                mask_path = os.path.join(kos_path, fname.replace(".jpg", "_label.bmp"))
                if os.path.exists(mask_path):
                    samples.append((img_path, mask_path))
    print(f"Total samples found: {len(samples)}")
    return samples


class CrackSegDataset(Dataset):
    def __init__(self, samples, size=(512, 1408), augment=True):
        self.samples = samples
        self.size    = size
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img  = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        img  = TF.resize(img,  self.size, interpolation=Image.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=Image.NEAREST)

        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            # Vertical flip
            if random.random() < 0.5:
                img  = TF.vflip(img)
                mask = TF.vflip(mask)
            # Random 90-degree rotation
            # if random.random() < 0.3:
            #    k = random.choice([1, 2, 3])
            #    img  = TF.rotate(img,  angle=90 * k)
            #    mask = TF.rotate(mask, angle=90 * k)
            # Color jitter (image only, not mask)
            if random.random() < 0.4:
                img = TF.adjust_brightness(img, brightness_factor=random.uniform(0.8, 1.2))
                img = TF.adjust_contrast(img,   contrast_factor=random.uniform(0.8, 1.2))

        img  = TF.to_tensor(img)                           # [3, H, W], float32 in [0,1]
        img  = TF.normalize(img, mean=[0.485, 0.456, 0.406],
                                  std= [0.229, 0.224, 0.225])

        mask_np = np.array(mask)
        mask_t  = torch.from_numpy((mask_np > 0).astype(np.float32))[None]  # [1, H, W]

        return img, mask_t

## Train/Validation Split

Split samples into positive and negative groups, then create a stratified training and validation split with a fixed seed.


In [ ]:
# ─────────────────────────────────────────────
# 4. Split — preserve positive/negative ratio
# ─────────────────────────────────────────────
def split_samples(samples, split_seed=SPLIT_SEED):
    rng = random.Random(split_seed)

    positive, negative = [], []
    for img_path, mask_path in samples:
        mask = np.array(Image.open(mask_path).convert("L"))
        if (mask > 0).any():
            positive.append((img_path, mask_path))
        else:
            negative.append((img_path, mask_path))

    print(f"Positive: {len(positive)}  |  Negative: {len(negative)}")

    pos_train, pos_val = train_test_split(
        positive, test_size=0.2, random_state=split_seed
    )
    neg_train, neg_val = train_test_split(
        negative, test_size=0.2, random_state=split_seed
    )

    # Cap negatives to 2× positives to avoid overwhelming the minority class
    neg_train = rng.sample(neg_train, min(len(neg_train), 2 * len(pos_train)))

    train_samples = pos_train + neg_train
    val_samples   = pos_val + neg_val

    rng.shuffle(train_samples)
    rng.shuffle(val_samples)

    print(f"Train: {len(train_samples)} ({len(pos_train)} pos + {len(neg_train)} neg)")
    print(f"Val:   {len(val_samples)}   ({len(pos_val)} pos + {len(neg_val)} neg)")

    return train_samples, val_samples

## Model Definition

Build the ResNet18 encoder with a UNet decoder using segmentation-models-pytorch and move it to the active device.


In [ ]:
# ─────────────────────────────────────────────
# 5. Model — segmentation-models-pytorch UNet
# ─────────────────────────────────────────────
def build_model():
    model = smp.Unet(
        encoder_name    = "resnet18",
        encoder_weights = "imagenet",
        in_channels     = 3,
        classes         = 1,
        activation      = None,   # raw logits — we apply sigmoid in the loss/metrics
    )
    return model.to(device)

## Fine-Tuning Strategy

Select which model parameters are trainable for full fine-tuning, decoder-only training, or partial adaptation.


In [ ]:
# ─────────────────────────────────────────────
# 5b. Adaptation strategy
# ─────────────────────────────────────────────
def apply_strategy(model, strategy_name="full_ft"):
    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    if strategy_name == "full_ft":
        for p in model.parameters():
            p.requires_grad = True

    elif strategy_name == "decoder_only":
        for p in model.decoder.parameters():
            p.requires_grad = True
        for p in model.segmentation_head.parameters():
            p.requires_grad = True

    elif strategy_name == "last_block_decoder":
        for p in model.decoder.parameters():
            p.requires_grad = True
        for p in model.segmentation_head.parameters():
            p.requires_grad = True

        # ResNet18 last encoder block
        for p in model.encoder.layer4.parameters():
            p.requires_grad = True

    elif strategy_name == "norm_only":
        for module in model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):
                for p in module.parameters():
                    p.requires_grad = True
        for p in model.segmentation_head.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Strategy: {strategy_name}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    return model

## Loss Functions

Define Dice, BCE, and focal loss components, then combine them according to the configured loss mode.


In [ ]:
# ─────────────────────────────────────────────
# 6. Loss
# ─────────────────────────────────────────────
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(2, 3))
    denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps)).mean()


_bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=device)
)

def focal_loss(logits, targets, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )

    probs = torch.sigmoid(logits)

    pt = probs * targets + (1 - probs) * (1 - targets)

    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)

    focal_weight = alpha_t * (1 - pt).pow(gamma)

    return (focal_weight * bce).mean()

def compute_loss_components(logits, targets):
    bce = _bce(logits, targets)
    dice = dice_loss(logits, targets)
    focal = focal_loss(logits, targets)

    if LOSS_MODE == "bce_dice":
        total = 0.5 * bce + 0.5 * dice
    elif LOSS_MODE == "focal_dice":
        total = 0.5 * focal + 0.5 * dice
    elif LOSS_MODE == "focal":
        total = focal
    else:
        raise ValueError(f"Unknown LOSS_MODE: {LOSS_MODE}")

    return total, bce, dice, focal

## Metrics

Compute segmentation metrics such as Dice, IoU, precision, recall, positive-only scores, and false positives on negative images.


In [ ]:
# ─────────────────────────────────────────────
# 7. Metrics
# ─────────────────────────────────────────────
def compute_batch_metrics(logits, targets, thr=THR, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > thr).float()

    tp = (preds * targets).sum(dim=(2, 3))
    fp = (preds * (1 - targets)).sum(dim=(2, 3))
    fn = ((1 - preds) * targets).sum(dim=(2, 3))

    gt_empty = (targets.sum(dim=(2, 3)) == 0)
    pred_empty = (preds.sum(dim=(2, 3)) == 0)
    pos_mask = ~gt_empty

    dice_all = []
    iou_all = []

    for i in range(targets.size(0)):
        if gt_empty[i]:
            # perfect if both are empty
            dice_all.append(1.0 if pred_empty[i] else 0.0)
            iou_all.append(1.0 if pred_empty[i] else 0.0)
        else:
            dice_i = (2 * tp[i] / (2 * tp[i] + fp[i] + fn[i] + eps)).item()
            iou_i  = (tp[i] / (tp[i] + fp[i] + fn[i] + eps)).item()
            dice_all.append(dice_i)
            iou_all.append(iou_i)

    precision = (tp / (tp + fp + eps)).mean().item()
    recall    = (tp / (tp + fn + eps)).mean().item()

    pos_dice = None
    pos_iou = None
    if pos_mask.any():
        pos_dice = (
            2 * tp[pos_mask] / (2 * tp[pos_mask] + fp[pos_mask] + fn[pos_mask] + eps)
        ).mean().item()
        pos_iou = (
            tp[pos_mask] / (tp[pos_mask] + fp[pos_mask] + fn[pos_mask] + eps)
        ).mean().item()

    neg_fp_rate = None
    if gt_empty.any():
        neg_fp_rate = (~pred_empty[gt_empty]).float().mean().item()

    return {
        "iou": float(np.mean(iou_all)),
        "dice": float(np.mean(dice_all)),
        "precision": precision,
        "recall": recall,
        "pos_iou": pos_iou,
        "pos_dice": pos_dice,
        "neg_fp_rate": neg_fp_rate,
    }

## Selection Score

Combine positive-image Dice and negative-image false-positive rate into the score used for model selection.


In [ ]:
# ─────────────────────────────────────────────
# 7b. Model selection score
# ─────────────────────────────────────────────
def model_selection_score(metrics, lam=MODEL_SELECTION_LAMBDA):
    pos_dice = metrics["pos_dice"] if metrics["pos_dice"] is not None else 0.0
    neg_fp   = metrics["neg_fp_rate"] if metrics["neg_fp_rate"] is not None else 0.0
    return pos_dice - lam * neg_fp

## Epoch Loop

Run one training or validation epoch, update model weights when training, and aggregate losses and metrics.


In [ ]:
# ─────────────────────────────────────────────
# 8. Train / Val loop
# ─────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, train=True, thr=THR):
    model.train() if train else model.eval()

    total_loss = 0.0
    total_bce = 0.0
    total_dice_loss = 0.0
    total_focal = 0.0

    all_iou, all_dice, all_prec, all_rec = [], [], [], []
    all_pos_iou, all_pos_dice, all_neg_fp = [], [], []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss, bce_val, dice_val, focal_val = compute_loss_components(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_bce += bce_val.item() * bs
        total_dice_loss += dice_val.item() * bs
        total_focal += focal_val.item() * bs

        metrics = compute_batch_metrics(logits.detach(), y.detach(), thr=thr)

        all_iou.append(metrics["iou"])
        all_dice.append(metrics["dice"])
        all_prec.append(metrics["precision"])
        all_rec.append(metrics["recall"])

        if metrics["pos_iou"] is not None:
            all_pos_iou.append(metrics["pos_iou"])
        if metrics["pos_dice"] is not None:
            all_pos_dice.append(metrics["pos_dice"])
        if metrics["neg_fp_rate"] is not None:
            all_neg_fp.append(metrics["neg_fp_rate"])

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "bce": total_bce / n,
        "dice_loss": total_dice_loss / n,
        "focal": total_focal / n,
        "iou": float(np.mean(all_iou)),
        "dice": float(np.mean(all_dice)),
        "precision": float(np.mean(all_prec)),
        "recall": float(np.mean(all_rec)),
        "pos_iou": float(np.mean(all_pos_iou)) if all_pos_iou else None,
        "pos_dice": float(np.mean(all_pos_dice)) if all_pos_dice else None,
        "neg_fp_rate": float(np.mean(all_neg_fp)) if all_neg_fp else None,
    }

## Training Routine

Train the model across epochs, update the scheduler, save the best checkpoint, and store metric history.


In [ ]:
# ─────────────────────────────────────────────
# 9. Training
# ─────────────────────────────────────────────
def train(model, train_loader, val_loader, checkpoint_path):
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    history = {k: [] for k in [
        "train_loss", "val_loss",
        "train_bce", "val_bce",
        "train_dice_loss", "val_dice_loss",
        "train_focal", "val_focal",
        "train_iou", "val_iou",
        "train_dice", "val_dice",
        "train_precision", "val_precision",
        "train_recall", "val_recall",
        "train_pos_iou", "val_pos_iou",
        "train_pos_dice", "val_pos_dice",
        "train_neg_fp_rate", "val_neg_fp_rate",
        "train_model_score", "val_model_score",
    ]}

    best_model_score = -1e9
    best_metrics = None
    start = time.time()

    metric_names = [
        "loss", "bce", "dice_loss", "focal",
        "iou", "dice", "precision", "recall",
        "pos_iou", "pos_dice", "neg_fp_rate"
    ]

    for epoch in range(1, NUM_EPOCHS + 1):
        tr = run_epoch(model, train_loader, optimizer, train=True, thr=THR)
        va = run_epoch(model, val_loader, optimizer=None, train=False, thr=THR)
        scheduler.step()

        tr_score = model_selection_score(tr)
        va_score = model_selection_score(va)

        for k in metric_names:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(va[k])

        history["train_model_score"].append(tr_score)
        history["val_model_score"].append(va_score)

        saved = ""
        if va_score > best_model_score:
            best_model_score = va_score
            torch.save(model.state_dict(), checkpoint_path)

            best_metrics = {
                "epoch": epoch,
                "loss": va["loss"],
                "bce": va["bce"],
                "dice_loss": va["dice_loss"],
                "focal": va["focal"],
                "iou": va["iou"],
                "dice": va["dice"],
                "precision": va["precision"],
                "recall": va["recall"],
                "pos_iou": va["pos_iou"],
                "pos_dice": va["pos_dice"],
                "neg_fp_rate": va["neg_fp_rate"],
                "model_score": va_score,
            }
            saved = "  ← saved"

        print(
            f"[{epoch:3d}/{NUM_EPOCHS}] "
            f"loss {tr['loss']:.4f}/{va['loss']:.4f} | "
            f"bce {tr['bce']:.4f}/{va['bce']:.4f} | "
            f"dice_loss {tr['dice_loss']:.4f}/{va['dice_loss']:.4f} | "
            f"dice {tr['dice']:.3f}/{va['dice']:.3f} | "
            f"pos_dice {(tr['pos_dice'] if tr['pos_dice'] is not None else float('nan')):.3f}/"
            f"{(va['pos_dice'] if va['pos_dice'] is not None else float('nan')):.3f} | "
            f"iou {tr['iou']:.3f}/{va['iou']:.3f} | "
            f"precision {tr['precision']:.3f}/{va['precision']:.3f} | "
            f"recall {tr['recall']:.3f}/{va['recall']:.3f} | "
            f"neg_fp {(tr['neg_fp_rate'] if tr['neg_fp_rate'] is not None else float('nan')):.3f}/"
            f"{(va['neg_fp_rate'] if va['neg_fp_rate'] is not None else float('nan')):.3f} | "
            f"score {tr_score:.4f}/{va_score:.4f}"
            f"{saved}"
        )

    elapsed = time.time() - start
    print(f"\nTraining complete in {elapsed/60:.1f} min | Best model score: {best_model_score:.4f}")

    return history, best_metrics

## Experiment Runner

Build dataloaders, apply the selected strategy, run one or more seeded experiments, and collect results.


In [ ]:
# ─────────────────────────────────────────────
# 9b. Experiment runners
# ─────────────────────────────────────────────
def run_single_experiment(train_samples, val_samples, run_seed, strategy_name="full_ft"):
    set_seed(run_seed)

    train_ds = CrackSegDataset(train_samples, size=IMG_SIZE, augment=True)
    val_ds   = CrackSegDataset(val_samples,   size=IMG_SIZE, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = build_model()
    model = apply_strategy(model, strategy_name=strategy_name)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    checkpoint_path = f"best_{strategy_name}_seed{run_seed}.pth"

    history, best_metrics = train(
        model,
        train_loader,
        val_loader,
        checkpoint_path=checkpoint_path
    )

    result = {
        "strategy": strategy_name,
        "seed": run_seed,
        "checkpoint_path": checkpoint_path,
        "best_epoch": best_metrics["epoch"],
        "best_val_loss": best_metrics["loss"],
        "best_val_iou": best_metrics["iou"],
        "best_val_dice": best_metrics["dice"],
        "best_val_precision": best_metrics["precision"],
        "best_val_recall": best_metrics["recall"],
        "best_val_pos_iou": best_metrics["pos_iou"],
        "best_val_pos_dice": best_metrics["pos_dice"],
        "best_val_neg_fp_rate": best_metrics["neg_fp_rate"],
        "best_model_score": best_metrics["model_score"],
        "trainable_params": trainable_params,
        "total_params": total_params,
    }

    return history, result


def run_multi_seed(strategy_name="full_ft", run_seeds=RUN_SEEDS):
    all_samples = collect_samples(DATA_ROOT)
    train_samples, val_samples = split_samples(all_samples, split_seed=SPLIT_SEED)

    histories = []
    results = []

    for seed in run_seeds:
        print(f"\n{'='*60}")
        print(f"Strategy: {strategy_name} | Seed: {seed}")
        print(f"{'='*60}")

        history, result = run_single_experiment(
            train_samples=train_samples,
            val_samples=val_samples,
            run_seed=seed,
            strategy_name=strategy_name
        )

        histories.append(history)
        results.append(result)

    return histories, results

## Result Summary

Aggregate numeric metrics across runs and print a compact summary of the experiment outcome.


In [ ]:
# ─────────────────────────────────────────────
# 11. Result summary
# ─────────────────────────────────────────────
def summarize_results(results):
    numeric_keys = [
        "best_val_loss",
        "best_val_iou",
        "best_val_dice",
        "best_val_precision",
        "best_val_recall",
        "best_val_pos_iou",
        "best_val_pos_dice",
        "best_val_neg_fp_rate",
        "best_model_score",
    ]

    summary = {}
    for key in numeric_keys:
        values = [r[key] for r in results if r[key] is not None]
        summary[key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
        }

    return summary


def print_summary(summary, strategy_name):
    print(f"\nSummary for strategy: {strategy_name}")
    print("-" * 50)
    for key, stats in summary.items():
        print(f"{key}: {stats['mean']:.4f} ± {stats['std']:.4f}")

## Learning Curves

Plot training and validation losses, metric curves, and model-selection score across epochs.


In [ ]:
# ─────────────────────────────────────────────
# 10. Learning curve plots
# ─────────────────────────────────────────────
def plot_history(history, strategy_name="unknown"):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"{strategy_name} — Learning Curves ({LOSS_MODE})", fontsize=14)

    # Total loss
    axes[0, 0].plot(epochs, history["train_loss"], label="Train total")
    axes[0, 0].plot(epochs, history["val_loss"], label="Val total")
    axes[0, 0].set_title("Total Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    # Loss components
    axes[0, 1].plot(epochs, history["train_bce"], label="Train BCE")
    axes[0, 1].plot(epochs, history["val_bce"], label="Val BCE")
    axes[0, 1].plot(epochs, history["train_dice_loss"], label="Train Dice loss", linestyle="--")
    axes[0, 1].plot(epochs, history["val_dice_loss"], label="Val Dice loss", linestyle="--")
    axes[0, 1].plot(epochs, history["train_focal"], label="Train Focal", linestyle=":")
    axes[0, 1].plot(epochs, history["val_focal"], label="Val Focal", linestyle=":")
    axes[0, 1].set_title("Loss Components")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    # Dice / IoU
    axes[1, 0].plot(epochs, history["train_dice"], label="Train Dice")
    axes[1, 0].plot(epochs, history["val_dice"], label="Val Dice")
    axes[1, 0].plot(epochs, history["train_pos_dice"], label="Train Pos Dice", linestyle="--")
    axes[1, 0].plot(epochs, history["val_pos_dice"], label="Val Pos Dice", linestyle="--")
    axes[1, 0].plot(epochs, history["train_iou"], label="Train IoU", linestyle=":")
    axes[1, 0].plot(epochs, history["val_iou"], label="Val IoU", linestyle=":")
    axes[1, 0].set_title("Segmentation Metrics")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    # Precision / Recall / Neg FP
    axes[1, 1].plot(epochs, history["train_precision"], label="Train Precision")
    axes[1, 1].plot(epochs, history["val_precision"], label="Val Precision")
    axes[1, 1].plot(epochs, history["train_recall"], label="Train Recall", linestyle="--")
    axes[1, 1].plot(epochs, history["val_recall"], label="Val Recall", linestyle="--")
    axes[1, 1].plot(epochs, history["train_neg_fp_rate"], label="Train Neg FP", linestyle=":")
    axes[1, 1].plot(epochs, history["val_neg_fp_rate"], label="Val Neg FP", linestyle=":")
    axes[1, 1].set_title("Detection Behavior")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Score")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("learning_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: learning_curves.png")

## Prediction Visualization

Load the best checkpoint and display validation images with ground-truth masks, predictions, and overlays.


In [ ]:
# ─────────────────────────────────────────────
# 12. Qualitative visualisation
# ─────────────────────────────────────────────
def visualise_predictions(model, val_loader, checkpoint_path, n=4, thr=THR, only_positive=True):
    """
    Show examples: image | ground truth | predicted mask | overlay.

    If only_positive=True, it tries to show validation images that actually contain cracks.
    This is important because the dataset has many negative samples.
    """
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    selected_imgs = []
    selected_gts = []
    selected_preds = []

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_dev = x_batch.to(device)
            logits = model(x_dev)
            probs = torch.sigmoid(logits).cpu().numpy()

            imgs = x_batch.numpy().transpose(0, 2, 3, 1)
            imgs = (imgs * std + mean).clip(0, 1)

            masks_gt = y_batch.numpy()[:, 0]
            masks_pred = probs[:, 0]

            for i in range(imgs.shape[0]):
                has_crack = masks_gt[i].sum() > 0

                if only_positive and not has_crack:
                    continue

                selected_imgs.append(imgs[i])
                selected_gts.append(masks_gt[i])
                selected_preds.append(masks_pred[i])

                if len(selected_imgs) >= n:
                    break

            if len(selected_imgs) >= n:
                break

    if len(selected_imgs) == 0:
        print("No positive validation samples found. Showing first batch instead.")
        x_batch, y_batch = next(iter(val_loader))

        x_dev = x_batch.to(device)
        with torch.no_grad():
            logits = model(x_dev)

        probs = torch.sigmoid(logits).cpu().numpy()

        imgs = x_batch.numpy().transpose(0, 2, 3, 1)
        imgs = (imgs * std + mean).clip(0, 1)

        selected_imgs = list(imgs)
        selected_gts = list(y_batch.numpy()[:, 0])
        selected_preds = list(probs[:, 0])

    n = min(n, len(selected_imgs))

    # Better figure size for wide 512x1408 images
    fig, axes = plt.subplots(n, 4, figsize=(24, 4 * n))

    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Image", "Ground truth", f"Prediction (thr={thr})", "Overlay"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=14)

    for i in range(n):
        img = selected_imgs[i]
        gt = selected_gts[i]
        pred_prob = selected_preds[i]
        pred_bin = pred_prob > thr

        axes[i, 0].imshow(img, aspect="auto")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt, cmap="gray", vmin=0, vmax=1, aspect="auto")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred_bin, cmap="gray", vmin=0, vmax=1, aspect="auto")
        axes[i, 2].axis("off")

        overlay = img.copy()

        tp = pred_bin & (gt > 0)
        fp = pred_bin & (gt == 0)
        fn = (~pred_bin) & (gt > 0)

        overlay[tp] = [0.0, 1.0, 0.0]  # true positive: green
        overlay[fp] = [1.0, 0.0, 0.0]  # false positive: red
        overlay[fn] = [0.0, 0.0, 1.0]  # false negative: blue

        axes[i, 3].imshow(overlay, aspect="auto")
        axes[i, 3].axis("off")

    plt.tight_layout()
    plt.savefig("predictions.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved: predictions.png")

## Threshold Sweep

Evaluate multiple prediction thresholds to see how validation performance and false positives change.


In [ ]:
# ─────────────────────────────────────────────
# 12. Thresholds sweep
# ─────────────────────────────────────────────
def evaluate_thresholds(model, val_loader, checkpoint_path, thresholds):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    results = []
    for thr in thresholds:
        metrics = run_epoch(model, val_loader, optimizer=None, train=False, thr=thr)
        results.append({
            "thr": thr,
            "val_loss": metrics["loss"],
            "val_dice": metrics["dice"],
            "val_pos_dice": metrics["pos_dice"],
            "val_iou": metrics["iou"],
            "val_pos_iou": metrics["pos_iou"],
            "val_neg_fp_rate": metrics["neg_fp_rate"],
            "val_model_score": model_selection_score(metrics),
        })

    print("\nThreshold sweep:")
    for r in results:
        print(
            f"thr={r['thr']:.2f} | "
            f"dice={r['val_dice']:.4f} | "
            f"pos_dice={r['val_pos_dice'] if r['val_pos_dice'] is not None else float('nan'):.4f} | "
            f"iou={r['val_iou']:.4f} | "
            f"neg_fp={r['val_neg_fp_rate'] if r['val_neg_fp_rate'] is not None else float('nan'):.4f} | "
            f"score={r['val_model_score']:.4f}"
        )

    best = max(results, key=lambda x: x["val_model_score"])
    print(f"\nBest threshold by model score: {best['thr']:.2f}")
    return results, best

## Main Execution

Choose the strategy, run training, summarize results, plot curves, rebuild validation data, visualize predictions, and run the threshold sweep.


In [ ]:
# ─────────────────────────────────────────────
# 13. Main
# ─────────────────────────────────────────────
if __name__ == "__main__":
    strategy = "full_ft"   # "decoder_only", "last_block_decoder", "norm_only", "full_ft"

    histories, results = run_multi_seed(strategy_name=strategy, run_seeds=RUN_SEEDS)
    summary = summarize_results(results)
    print_summary(summary, strategy)

    # Plot first seed history
    plot_history(histories[0], strategy_name=strategy)

    # Rebuild val loader for visualization / threshold sweep
    all_samples = collect_samples(DATA_ROOT)
    train_samples, val_samples = split_samples(all_samples, split_seed=SPLIT_SEED)

    val_ds = CrackSegDataset(val_samples, size=IMG_SIZE, augment=False)
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = build_model()
    model = apply_strategy(model, strategy)
    best_result = max(results, key=lambda r: r["best_model_score"])
    checkpoint_path = best_result["checkpoint_path"]

    print(f"\nUsing checkpoint from seed {best_result['seed']} for visualization/threshold sweep")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Best model score: {best_result['best_model_score']:.4f}")

    visualise_predictions(
    model,
    val_loader,
    checkpoint_path=checkpoint_path,
    n=4,
    thr=THR,
    only_positive=True
    )

    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    threshold_results, best_thr = evaluate_thresholds(
        model,
        val_loader,
        checkpoint_path=checkpoint_path,
        thresholds=thresholds
    )